In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
import json
from sklearn.linear_model import LogisticRegression


In [2]:

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


In [3]:
# Load LSTM bits
with open('../../models/lstm_v2_meta.json', 'r') as f:
    meta = json.load(f)

In [4]:

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim, dropout=0.3):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout) # Attention dropout added

    def forward(self, lstm_out):
        scores = self.attn(lstm_out)           
        weights = torch.softmax(scores, dim=1) 
        weights = self.dropout(weights) # Apply dropout to attention weights
        context = (weights * lstm_out).sum(dim=1)  
        return context, weights.squeeze(-1)

class LeptoLSTM_v2(nn.Module):
    def __init__(self, seq_input_dim, static_input_dim, hidden_dim=128, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=seq_input_dim, out_channels=seq_input_dim * 2, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=seq_input_dim * 2,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        bi_dim = hidden_dim * 2
        self.layer_norm = nn.LayerNorm(bi_dim) # LayerNorm added after LSTM

        self.attention = TemporalAttention(bi_dim, dropout=dropout)

        self.static_branch = nn.Sequential(
            nn.Linear(static_input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.Linear(bi_dim + 32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_stat):
        x_seq_c = x_seq.transpose(1, 2)
        x_seq_c = self.conv(x_seq_c)
        x_seq_c = self.relu(x_seq_c)
        x_seq_c = self.conv_dropout(x_seq_c)
        
        x_seq_lstm = x_seq_c.transpose(1, 2)
        lstm_out, _ = self.lstm(x_seq_lstm)
        
        # Apply LayerNorm
        lstm_out = self.layer_norm(lstm_out)
        
        context, _ = self.attention(lstm_out)
        static_out  = self.static_branch(x_stat)
        combined    = torch.cat([context, static_out], dim=1)
        return self.head(combined)


### Loading necessary Artifacts from previous models

In [5]:

model_l = LeptoLSTM_v2(meta['seq_input_dim'], meta['static_input_dim'])
model_l.load_state_dict(torch.load('../../models/lstm_v2_model.pth', map_location='cpu'))
model_l.eval()

sc_seq = joblib.load('../../models/lstm_v2_scaler_seq.pkl')
sc_stat = joblib.load('../../models/lstm_v2_scaler_stat.pkl')

# Load RF bits
model_r = joblib.load('../../models/rf_model.pkl')
le = joblib.load('../../models/label_encoder.pkl')
drr_df = pd.read_csv('../../models/district_risk_rate.csv')
drr_map = dict(zip(drr_df.iloc[:,0], drr_df.iloc[:,1]))

# Prepare Aligned Data
test_df = pd.read_csv('../../data/processed/splitteddataset/lepto_monthly_test.csv', parse_dates=['YearMonth'])
test_df['District_enc'] = le.transform(test_df['District'])
test_df['District_risk_rate'] = test_df['District'].map(drr_map).astype(float)

In [6]:

def build_aligned(df, meta):
    X_sq, X_st, y_out, al_idx = [], [], [], []
    for d, g in df.groupby('District', observed=True):
        g = g.sort_values('YearMonth')
        idx = g.index.tolist()
        sq_v = g[meta['sequence_cols']].values.astype(np.float32)
        st_v = g[meta['static_cols']].values.astype(np.float32)
        lb = g['RiskLabel'].values
        for i in range(meta['seq_len'] - 1, len(g)):
            X_sq.append(sq_v[i - meta['seq_len'] + 1 : i + 1])
            X_st.append(st_v[i])
            y_out.append(lb[i])
            al_idx.append(idx[i])
    return np.array(X_sq), np.array(X_st), np.array(y_out), al_idx

X_q, X_s, y_t, idx_a = build_aligned(test_df, meta)


### Predictions for RF+ LSTM separately

In [7]:

# LSTM Prediction
X_q_s = sc_seq.transform(X_q.reshape(-1, X_q.shape[2])).reshape(X_q.shape)
X_s_s = sc_stat.transform(X_s)
with torch.no_grad():
    probs_l = torch.sigmoid(model_l(torch.tensor(X_q_s), torch.tensor(X_s_s))).flatten().numpy()

# RF Prediction
test_al = test_df.iloc[idx_a].copy()
X_rf = test_al.drop(columns=['District', 'RiskLabel', 'YearMonth'])

# Ensure X_rf features are exactly what model_r expects
if hasattr(model_r, 'feature_names_in_'):
    X_rf = X_rf[model_r.feature_names_in_]
probs_r = model_r.predict_proba(X_rf)[:, 1]

### Stacking Meta-Learner (Trained on full train set OOF(Out‑Of‑Fold)/predictions)

In [8]:
train_df = pd.read_csv('../../data/processed/splitteddataset/lepto_monthly_train.csv', parse_dates=['YearMonth'])
train_df['District_enc'] = le.fit_transform(train_df['District']) # fit_transform, though le is already fitted
train_df['District_risk_rate'] = train_df['District'].map(drr_map).astype(float)
X_q_tr, X_s_tr, y_tr, idx_a_tr = build_aligned(train_df, meta)

X_q_s_tr = sc_seq.transform(X_q_tr.reshape(-1, X_q_tr.shape[2])).reshape(X_q_tr.shape)
X_s_s_tr = sc_stat.transform(X_s_tr)
with torch.no_grad():
    probs_l_tr = torch.sigmoid(model_l(torch.tensor(X_q_s_tr), torch.tensor(X_s_s_tr))).flatten().numpy()

train_al = train_df.iloc[idx_a_tr].copy()
X_rf_tr = train_al.drop(columns=['District', 'RiskLabel', 'YearMonth'])
if hasattr(model_r, 'feature_names_in_'): X_rf_tr = X_rf_tr[model_r.feature_names_in_]
probs_r_tr = model_r.predict_proba(X_rf_tr)[:, 1]

# Ensemble Prediction using Static Weighted Averaging (Option 2)
# We use 0.6 weight for the robust LSTM, and 0.4 weight for the RF to reduce overfitting.
w_lstm = 0.6
w_rf = 0.4

probs_e = w_lstm * probs_l + w_rf * probs_r
probs_e_tr = w_lstm * probs_l_tr + w_rf * probs_r_tr

auc_e = roc_auc_score(y_t, probs_e)
auc_tr = roc_auc_score(y_tr, probs_e_tr)
gap = auc_tr - auc_e

print(f'Train AUC: {auc_tr:.4f}')
print(f'Test AUC: {auc_e:.4f}')
print(f'AUC Gap: {gap:.4f}')

Train AUC: 0.9614
Test AUC: 0.9121
AUC Gap: 0.0493


### Summary

In [9]:
thresh = 0.45
def calc_metrics(probs, y):
    preds = (probs >= thresh).astype(int)
    tp = ((preds == 1) & (y == 1)).sum()
    tn = ((preds == 0) & (y == 0)).sum()
    fp = ((preds == 1) & (y == 0)).sum()
    fn = ((preds == 0) & (y == 1)).sum()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1 = 2 * prec * sens / (prec + sens + 1e-9)
    return roc_auc_score(y, probs), average_precision_score(y, probs), sens, spec, f1

l_m = calc_metrics(probs_l, y_t)
r_m = calc_metrics(probs_r, y_t)
e_m = calc_metrics(probs_e, y_t)

res = pd.DataFrame({
    'Metric': ['ROC-AUC', 'AP', 'Sens', 'Spec', 'F1 HR'],
    'LSTM v2': l_m,
    'RF': r_m,
    'Ensemble': e_m
})
print('\nComparison Table:')
print(res)


Comparison Table:
    Metric   LSTM v2        RF  Ensemble
0  ROC-AUC  0.902025  0.905846  0.912139
1       AP  0.907975  0.913172  0.917092
2     Sens  0.855856  0.822072  0.835586
3     Spec  0.760915  0.817048  0.821206
4    F1 HR  0.809372  0.813824  0.823529
